### Preprocessing Notebook Overview

This notebook prepares the dunnhumby transaction data for household-level customer segmentation. The goal is to turn raw transaction rows into one clean dataset where each row represents one household and summarizes its shopping behavior.

#### Import Libraries

This section imports the main Python libraries used for data cleaning, analysis, and visualization. These libraries will support the transaction-level preprocessing and household-level feature creation.

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Improve plot styling
sns.set()

# Show all columns during EDA
pd.set_option('display.max_columns', None)

#### Define File Paths

This section sets up the project folder paths used to load raw data and save cleaned outputs. Keeping paths organized makes it easier to move between notebooks without rewriting file locations.

In [2]:
# File path locations
from pathlib import Path

# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

#### Import Transaction Data
This section loads the raw dunnhumby transaction table and takes an initial look at the data. Since this project starts with transaction-only segmentation, this table is the main source for household shopping behavior.

In [3]:
transactions = pd.read_csv(RAW_DATA_DIR / "transaction_data.csv")
transactions.head()

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


#### Initial Data Checks
This section reviews the shape, column names, data types, missing values, and basic summary statistics of the transaction data. These checks help confirm the data is usable before creating household-level features.

In [4]:
transactions.shape

(2595732, 12)

In [5]:
# Standardize column names
transactions.columns = (
    transactions.columns.str.strip()
             .str.lower()
             .str.replace(" ", "_")
             .str.replace("-", "_")
)

In [6]:
transactions_types_null_counts = pd.DataFrame()
transactions_types_null_counts["Data Type"] = transactions.dtypes
transactions_types_null_counts["Count of Nulls"] = transactions.isna().sum()
transactions_types_null_counts

,Data Type,Count of Nulls
household_key,int64,0
basket_id,int64,0
day,int64,0
product_id,int64,0
quantity,int64,0
sales_value,float64,0
store_id,int64,0
retail_disc,float64,0
trans_time,int64,0
week_no,int64,0


In [7]:
transactions.describe()

,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
count,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06
mean,1.271953e+03,3.402620e+10,3.887562e+02,2.891435e+06,1.004286e+02,3.104120e+00,3.142673e+03,-5.387054e-01,1.561586e+03,5.622150e+01,-1.641600e-02,-2.918564e-03
std,7.260660e+02,4.711649e+09,1.897210e+02,3.837404e+06,1.153436e+03,4.182274e+00,8.937113e+03,1.249191e+00,3.998378e+02,2.710223e+01,2.168410e-01,3.969004e-02
min,1.000000e+00,2.698485e+10,1.000000e+00,2.567100e+04,0.000000e+00,0.000000e+00,1.000000e+00,-1.800000e+02,0.000000e+00,1.000000e+00,-5.593000e+01,-7.700000e+00
25%,6.560000e+02,3.040805e+10,2.290000e+02,9.174590e+05,1.000000e+00,1.290000e+00,3.300000e+02,-6.900000e-01,1.308000e+03,3.300000e+01,0.000000e+00,0.000000e+00
50%,1.272000e+03,3.276081e+10,3.900000e+02,1.028816e+06,1.000000e+00,2.000000e+00,3.720000e+02,-1.000000e-02,1.613000e+03,5.600000e+01,0.000000e+00,0.000000e+00
75%,1.913000e+03,4.012685e+10,5.530000e+02,1.133018e+06,1.000000e+00,3.490000e+00,4.220000e+02,0.000000e+00,1.843000e+03,8.000000e+01,0.000000e+00,0.000000e+00
max,2.500000e+03,4.230536e+10,7.110000e+02,1.831630e+07,8.963800e+04,8.400000e+02,3.428000e+04,3.990000e+00,2.359000e+03,1.020000e+02,0.000000e+00,0.000000e+00


#### Households and Shopping Trips
This section checks how many unique households and shopping trips are present in the transaction data. These counts confirm that the dataset can support household-level segmentation based on shopping activity.

In [8]:
# Number of unique households
transactions["household_key"].nunique()

2500

In [9]:
# Number of unique shopping trips
transactions["basket_id"].nunique()

276484

#### Transaction Time Period

This section reviews the transaction time, day, and week fields to understand the time span of the data. The dataset covers roughly two years of shopping activity, which supports recency and shopping cadence features.

In [10]:
# Transaction times
print("Min transaction time", min(transactions["trans_time"]))
print("Max transaction time", max(transactions["trans_time"]))

Min transaction time 0
Max transaction time 2359


In [11]:
# Transaction days
print("Min day", min(transactions["day"]))
print("Max day", max(transactions["day"]))

Min day 1
Max day 711


In [12]:
# Transaction week number
print("Min week_no", min(transactions["week_no"]))
print("Max week_no", max(transactions["week_no"]))

Min week_no 1
Max week_no 102


#### Household Shopping Window
This section identifies each household’s first and most recent shopping day in the dataset. These values are used to create recency and average days between trips.

In [13]:
# Each household's first and last basket/shopping event
min_max_days = transactions.groupby("household_key").agg(
    min_day=("day" , "min"),
    max_day=("day" , "max")
)

min_max_days

,min_day,max_day
household_key,,
1,51,706
2,103,668
3,113,703
4,104,627
5,85,703
...,...,...
2496,117,683
2497,78,700
2498,105,710


In [14]:
# Confirm that all the time is the same for every basket_id
min_max_time = transactions.groupby(["household_key", "basket_id"]).agg(
    min_time=("trans_time" , "min"),
    max_time=("trans_time" , "max")
)

min_max_time[(min_max_time["min_time"]) != (min_max_time["max_time"])].shape[0]

0

#### Number of Trips per Household
This section calculates how many unique shopping trips each household made.

In [15]:
# Number of shopping baskets per household
households = transactions.groupby("household_key").agg(
    num_trips=("basket_id", "nunique")
)

households.head(10)

,num_trips
household_key,
1,86
2,45
3,47
4,30
5,40
6,250
7,59
8,113
9,20


#### Recency
This section calculates how recently each household shopped compared with the last day in the dataset. Recency is useful for identifying currently active households versus less recent shoppers.

In [16]:
# Recency
day_max = max(transactions["day"])
day_max

min_max_days["recency_days"] = day_max - min_max_days["max_day"]
min_max_days.head(10)

,min_day,max_day,recency_days
household_key,,,
1,51,706,5
2,103,668,43
3,113,703,8
4,104,627,84
5,85,703,8
6,118,707,4
7,23,709,2
8,65,706,5
9,104,689,22


#### Basket-Level Sales
This section aggregates transaction rows into basket-level sales totals. This step is needed because each transaction row represents an item-level purchase.

In [17]:
# Total value for each basket for each household
basket_sales = transactions.groupby(["household_key", "basket_id"]).agg(
    total_basket_sales_value=("sales_value", "sum")
)

basket_sales

total_basket_sales_value
household_key basket_id                            
1             27601281299                     78.66
              27774192959                     41.10
              28024266849                     26.90
              28106322445                     63.43
              28235481967                     53.45
...                                             ...
2500          42076591137                      4.00
              42115225458                     50.54
              42198526427                     73.65
              42229520973                     13.95
              42250641157                     45.12

[276484 rows x 1 columns]

#### Total Household Spend

This section calculates each household's total spend across all shopping trips.

In [18]:
# Total spend
total_spend_per_hh = basket_sales.groupby("household_key").agg(
    total_spend=("total_basket_sales_value", "sum")
)

total_spend_per_hh

,total_spend
household_key,
1,4330.16
2,1954.34
3,2653.21
4,1200.11
5,779.06
...,...
2496,4339.66
2497,7111.98
2498,2601.60


#### Average Basket Value
This section calculates the average basket value for each household by averaging their basket-level sales totals. This helps separate households with larger shopping trips from those with smaller baskets.

In [19]:
# Average 
avg_basket_sales_value = basket_sales.groupby("household_key").agg(
    avg_basket_value=("total_basket_sales_value", "mean")
).round(2)

avg_basket_sales_value.head(10)

,avg_basket_value
household_key,
1,50.35
2,43.43
3,56.45
4,40.00
5,19.48
6,23.98
7,57.63
8,48.98
9,39.87


#### Total Units Purchased
This section calculates the total number of units purchased by each household. 

In [20]:
# Total Units Purchased
total_units_purchased = transactions.groupby("household_key").agg(
    total_units_purchased=("quantity", "sum")
)

total_units_purchased.head(10)

,total_units_purchased
household_key,
1,1997
2,834
3,8540
4,382
5,245
6,94690
7,1554
8,68333
9,353


#### Combine Household Features
This section combines the household-level features into one dataset. The result is a single table where each row represents one household and each column describes part of its shopping behavior.

In [21]:
# Merge all dataframes
dfs_to_merge = [ 
    min_max_days, 
    total_spend_per_hh, 
    avg_basket_sales_value, 
    total_units_purchased
]

households_all_data = households.join(dfs_to_merge, how="left")

households_all_data.head(10)

,num_trips,min_day,max_day,recency_days,total_spend,avg_basket_value,total_units_purchased
household_key,,,,,,,
1,86,51,706,5,4330.16,50.35,1997
2,45,103,668,43,1954.34,43.43,834
3,47,113,703,8,2653.21,56.45,8540
4,30,104,627,84,1200.11,40.00,382
5,40,85,703,8,779.06,19.48,245
6,250,118,707,4,5996.16,23.98,94690
7,59,23,709,2,3400.05,57.63,1554
8,113,65,706,5,5534.97,48.98,68333
9,20,104,689,22,797.42,39.87,353


#### Average Days Between Trips
This section estimates how many days typically pass between each household's shopping trips. This gives a simple view of shopping cadence and helps distinguish routine shoppers from less frequent shoppers.

In [22]:
# Frequency
households_all_data["avg_days_between_trips"] = (
    households_all_data["max_day"] - households_all_data["min_day"]
) / (households_all_data["num_trips"] - 1)

households_all_data["avg_days_between_trips"] = households_all_data["avg_days_between_trips"].round(1)

households_all_data.head(10)

,num_trips,min_day,max_day,recency_days,total_spend,avg_basket_value,total_units_purchased,avg_days_between_trips
household_key,,,,,,,,
1,86,51,706,5,4330.16,50.35,1997,7.7
2,45,103,668,43,1954.34,43.43,834,12.8
3,47,113,703,8,2653.21,56.45,8540,12.8
4,30,104,627,84,1200.11,40.00,382,18.0
5,40,85,703,8,779.06,19.48,245,15.8
6,250,118,707,4,5996.16,23.98,94690,2.4
7,59,23,709,2,3400.05,57.63,1554,11.8
8,113,65,706,5,5534.97,48.98,68333,5.7
9,20,104,689,22,797.42,39.87,353,30.8


#### Average Basket Size
This section calculates the average number of units purchased per trip for each household. This helps identify households that tend to make larger stock-up trips versus smaller shopping trips.

In [23]:
households_all_data["avg_basket_size"] = (
    households_all_data["total_units_purchased"] / households_all_data["num_trips"]
).round(1)
households_all_data.head(10)

,num_trips,min_day,max_day,recency_days,total_spend,avg_basket_value,total_units_purchased,avg_days_between_trips,avg_basket_size
household_key,,,,,,,,,
1,86,51,706,5,4330.16,50.35,1997,7.7,23.2
2,45,103,668,43,1954.34,43.43,834,12.8,18.5
3,47,113,703,8,2653.21,56.45,8540,12.8,181.7
4,30,104,627,84,1200.11,40.00,382,18.0,12.7
5,40,85,703,8,779.06,19.48,245,15.8,6.1
6,250,118,707,4,5996.16,23.98,94690,2.4,378.8
7,59,23,709,2,3400.05,57.63,1554,11.8,26.3
8,113,65,706,5,5534.97,48.98,68333,5.7,604.7
9,20,104,689,22,797.42,39.87,353,30.8,17.6


#### Final Dataset Checks
This section checks the final household-level dataset for shape, missing values, summary statistics, and duplicate household records.

In [24]:
households_all_data.shape

(2500, 9)

In [25]:
households_all_data.isna().sum()

num_trips                 0
min_day                   0
max_day                   0
recency_days              0
total_spend               0
avg_basket_value          0
total_units_purchased     0
avg_days_between_trips    3
avg_basket_size           0
dtype: int64

In [26]:
households_all_data.describe()

,num_trips,min_day,max_day,recency_days,total_spend,avg_basket_value,total_units_purchased,avg_days_between_trips,avg_basket_size
count,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2.500000e+03,2497.000000,2500.000000
mean,110.593600,65.941200,685.426000,25.574000,3222.985232,31.620640,1.042742e+05,15.104646,726.704200
std,115.669368,39.152355,62.790655,62.790655,3349.026076,19.054935,2.444561e+05,24.706482,1239.624618
min,1.000000,1.000000,54.000000,0.000000,8.170000,2.390000,5.000000e+00,0.500000,1.600000
25%,39.000000,36.000000,691.000000,1.000000,970.740000,18.327500,7.700000e+02,4.500000,13.500000
50%,79.000000,69.000000,705.000000,6.000000,2157.750000,27.420000,1.108450e+04,8.100000,137.450000
75%,142.250000,95.000000,710.000000,20.000000,4413.320000,40.547500,8.764000e+04,16.400000,965.750000
max,1300.000000,671.000000,711.000000,657.000000,38319.790000,165.830000,4.479917e+06,589.000000,16322.500000


In [27]:
# Check that no household numbers repeat
households_all_data.index.is_unique

True

#### Handle Missing Trip Cadence Values
A few households only shopped once, so average days between trips could not be calculated for them. These missing values are filled with the full observation window to reflect very infrequent shopping behavior.

In [28]:
# Fill missing 'avg_days_between_trips'
households_all_data["avg_days_between_trips"] = households_all_data["avg_days_between_trips"].fillna(day_max)
households_all_data.isna().sum()

num_trips                 0
min_day                   0
max_day                   0
recency_days              0
total_spend               0
avg_basket_value          0
total_units_purchased     0
avg_days_between_trips    0
avg_basket_size           0
dtype: int64

#### Remove Helper Columns
The `min_day` and `max_day` columns were useful for creating recency and trip cadence features, but they are not needed in the final modeling dataset.

In [29]:
households_all_data = households_all_data.drop(["min_day", "max_day"], axis=1).copy()
households_all_data.columns

Index(['num_trips', 'recency_days', 'total_spend', 'avg_basket_value',
       'total_units_purchased', 'avg_days_between_trips', 'avg_basket_size'],
      dtype='object')

#### Export Clean Household Dataset
This section exports the final household-level shopping dataset for use in the next notebook. The output will be used for exploratory analysis and later clustering.

In [30]:
households_all_data.reset_index().to_parquet(
    CLEAN_DATA_DIR / "01_household_shopping.parquet", index=False
)